# KION AutoFE: прогон на Colab T4
Runtime → Change runtime type → **T4 GPU**. Долгие этапы идут в фоне (`nohup`), прогресс — ячейкой статуса.
`results/` и `logs/` лежат на Google Drive: после обрыва сессии повторный запуск пропускает готовые шаги.

In [ ]:
!nvidia-smi -L
!test -d /content/kion-autofe || git clone -q https://github.com/r3dmistyle/kion-autofe.git /content/kion-autofe
%cd /content/kion-autofe
!git pull -q
!pip -q install duckdb catboost tabulate
!python -c "import numpy as np; from catboost import CatBoostClassifier as C; X=np.random.rand(2000,5); C(iterations=20, task_type='GPU', verbose=0).fit(X, X[:,0]>.5); print('catboost GPU ok')"
!python tests/test_leakage.py && python src/ranker.py && python src/search.py

In [ ]:
# results and logs on Drive, so they survive a disconnect
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/kion-autofe/results /content/drive/MyDrive/kion-autofe/logs
!rm -rf results logs && ln -s /content/drive/MyDrive/kion-autofe/results results && ln -s /content/drive/MyDrive/kion-autofe/logs logs && ls -la results logs

In [ ]:
# smoke: every step on 5k users with a tiny budget (~10 min)
!nohup nice -n 10 python src/experiments.py --n-users 5000 --device gpu --seeds 1 --train-weeks 1 --pool 5 --mut-fits 2 > logs/smoke.log 2>&1 &

In [ ]:
# status
!for f in logs/*.log; do echo "--- $f"; grep -vE 'Warning|warn|df\[n\]|^ *$' $f | tail -6; done; ls results/*/; nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader; free -g | head -2

In [ ]:
# main run (~1.4 h on T4): baseline, hand-crafted, AutoFE search on 4k users per fold, comparison, window sweep
!nohup nice -n 10 python src/experiments.py --n-users 100000 --device gpu --seeds 2 --search-users 4000 --pool 60 --mut-fits 30 > logs/n100000.log 2>&1 &

In [ ]:
# scaling: same pipeline, features selected at 100k reused
!nohup sh -c 'for n in 25000 50000; do nice -n 10 python src/experiments.py --n-users $n --device gpu --seeds 1 --steps e0,hand,auto --search-from results/n100000 > logs/n$n.log 2>&1; done' > /dev/null 2>&1 &

In [ ]:
!python src/experiments.py --steps report
!zip -qr /content/results.zip -r results/ logs/ && ls -la /content/results.zip